# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Access and print metadata using the .metadata object (not subscripting)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\nPublished: {meta.datePublished}\nLicense: {meta.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their @ids
print('Available Record Sets:')
for rs in dataset.metadata.record_sets():
    print(f"- @id: {rs['@id']}, name: {rs.get('name', '(no name)')}, description: {rs.get('description', '(no description)')}")

In [ ]:
# List fields for all record sets, referenced by @id
from pprint import pprint

for rs in dataset.metadata.record_sets():
    print(f"\nRecord set @id: {rs['@id']}")
    if 'field' in rs:
        print('  Fields:')
        for field in rs['field']:
            field_obj = dataset.metadata.field_by_id(field['@id']) if isinstance(field, dict) else dataset.metadata.field_by_id(field)
            print(f"    - @id: {field_obj['@id']}, name: {field_obj.get('name')}, dataType: {field_obj.get('dataType')} ")
    else:
        print('  No fields in this record set.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For demonstration, extract data from the main record set.
# You may update record_set_ids as required after reviewing the previous cell's output.
# 1. Find all record set @ids.
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets()]
dataframes = {}

for record_set_id in record_set_ids:
    # Each record is a dict mapping field @id to value
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f'Record set {record_set_id} loaded. Shape: {df.shape}')
        print(f'Columns (@id): {df.columns.tolist()}')
        display(df.head())
    else:
        print(f'Record set {record_set_id} contains zero records.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# --- Example EDA: Update the variable values based on your exploration above ---
# Set variables for record set, fields, etc. using @id strings (from earlier cell output)
# We'll use the first non-empty dataframe, and choose numeric fields by inspection
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # Try to automatically find a likely numeric field by searching for floats/ints in first row
    numeric_field = None
    for col in df.columns:
        # Try to infer type for EDA
        try:
            if df[col].dtype in ['float64', 'int64']:
                numeric_field = col
                break
            # Try to cast for the first entry
            sample = df[col].dropna().iloc[0] if not df[col].dropna().empty else None
            float(sample)
            numeric_field = col
            df[col] = pd.to_numeric(df[col], errors='coerce')
            break
        except Exception:
            continue
    if numeric_field is None:
        print("No numeric fields found for EDA.")
    else:
        print(f"Using numeric field: {numeric_field}")

        # Filter: choose a quantile or threshold to filter
        threshold = df[numeric_field].quantile(0.75)
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()

        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by another field (categorical, e.g. string/object columns)
        group_field = None
        for col in df.columns:
            if col != numeric_field:
                if df[col].dtype == 'O':
                    group_field = col
                    break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable categorical grouping field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field is not None:
    # Histogram
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in record set '{record_set_id}'")
    plt.xlabel(numeric_field)
    plt.show()
    
    # If a group field was found earlier
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, we have demonstrated loading and exploring a FAIR-compliant Croissant dataset describing human mass spectrometry protein analysis. We reviewed record set structure, inspected field `@id`s, loaded data using `mlcroissant`, conducted basic EDA, and visualized one numeric distribution. This workflow can be extended for downstream biomarker and proteomics analyses, leveraging the semantic transparency of Croissant schemas.*